# ساخت برنامه‌های تولید تصاویر (Azure OpenAI)

مدل‌های زبان بزرگ (LLM) بیش از تولید متن کاربرد دارند. شما همچنین می‌توانید تصاویر را از توضیحات متنی تولید کنید. تصاویر به عنوان یک واسطه کاربردی در حوزه‌های پزشکی، معماری، گردشگری، توسعه بازی، بازاریابی و غیره بسیار مفید هستند. در این درس، ما با مدل‌های **GPT Image** امروزی در Microsoft Foundry کار می‌کنیم.

## اهداف یادگیری

در پایان این درس قادر خواهید بود:

- توضیح دهید تولید تصویر چیست و در کجا کاربرد دارد.
- خانواده مدل `gpt-image` را درک کنید و تفاوت آن با مدل‌های قدیمی DALL·E را بدانید.
- یک برنامه تولید تصویر بسازید و تصاویر را ویرایش کنید.

## تولید تصویر چیست؟

مدل‌های تولید تصویر تصاویر را از روی یک متن اولیه (prompt) می‌سازند. مدل‌های مدرن مانند `gpt-image` در طول آموزش رابطه بین متن و تصویر را یاد می‌گیرند و سپس به طور تدریجی نویز تصادفی را به تصویری تبدیل می‌کنند که با متن اولیه شما مطابقت دارد.

دو خانواده شناخته شده مدل تصویر عبارتند از:

- **`gpt-image` (OpenAI)** — نسل فعلی که در این درس استفاده می‌شود. این مدل تولید تصویر از متن و ویرایش تصویر (نقاشی درون ماسک) را پشتیبانی می‌کند.
- **Midjourney** — یک مدل محبوب شخص ثالث با سرویس و روند کاری مبتنی بر Discord خود.

> مدل‌های قدیمی‌تر تصویر OpenAI — **DALL·E 2** و **DALL·E 3** — مدل‌های قدیمی هستند. DALL·E 3 دیگر برای استقرارهای جدید در دسترس نیست و ویژگی‌هایی مانند `create_variation` فقط در DALL·E 2 وجود داشتند. برای برنامه‌های جدید از مدل‌های `gpt-image` استفاده کنید.

در Microsoft Foundry، مدل **`gpt-image-2`** جدیدترین و قدرتمندترین مدل تصویر است و به عنوان پیش‌فرض توصیه شده است. مدل‌های `gpt-image-1.5` و `gpt-image-1-mini` نیز به طور عمومی در دسترس هستند.

> **مهم:** مدل‌های `gpt-image` تصویر تولید شده را به صورت **base64** (`b64_json`) برمی‌گردانند، نه به صورت URL. کد شما رشته base64 را به بایت تبدیل کرده و ذخیره می‌کند — هیچ URL تصویری برای دانلود وجود ندارد.


## ساخت اولین برنامه تولید تصویر شما

پس ساخت یک برنامه تولید تصویر چه مواردی لازم است؟ شما به کتابخانه‌های زیر نیاز دارید:

- **python-dotenv**، به شدت توصیه می‌شود از این کتابخانه برای نگه داشتن اسرار خود در فایل *.env* دور از کد استفاده کنید.
- **openai**، این کتابخانه همان است که برای تعامل با API اپن‌ای‌آی استفاده خواهید کرد.
- **pillow**، برای کار با تصاویر در پایتون.

اگر قبلاً انجام نشده، دستورالعمل‌های صفحه [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) را دنبال کنید تا یک منبع و مدل Microsoft Foundry ایجاد کنید. مدل **gpt-image-2** را به عنوان مدل انتخاب کنید (جدیدترین مدل تصویر Azure OpenAI؛ DALL·E مدل قدیمی است).

1. یک فایل *.env* با محتوای زیر ایجاد کنید:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    این اطلاعات را در پرتال Microsoft Foundry برای منبع خود در بخش "Deployments" پیدا کنید.



۱. کتابخانه‌های بالا را در فایلی به نام *requirements.txt* جمع‌آوری کنید، به این صورت:

    ```text
    python-dotenv
    openai
    pillow
    ```


۱. سپس، محیط مجازی بسازید و کتابخانه‌ها را نصب کنید:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> برای ویندوز، از دستورات زیر برای ایجاد و فعال‌سازی محیط مجازی خود استفاده کنید:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. کد زیر را در فایلی به نام *app.py* اضافه کنید:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # وارد کردن dotenv
    dotenv.load_dotenv()

    # پیکربندی کلاینت Azure OpenAI (Microsoft Foundry).
    # مدل‌های تصویر به نسخه جدید API نیاز دارند — مستندات Foundry را برای نسخه مورد نیاز مدل خود بررسی کنید.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # ایجاد تصویر با استفاده از API تولید تصویر
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # متن درخواست خود را اینجا وارد کنید
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # مثلاً "gpt-image-2"
        )
        # تعیین مسیر دایرکتوری برای ذخیره تصویر
        image_dir = os.path.join(os.curdir, 'images')

        # اگر دایرکتوری وجود ندارد، آن را ایجاد کنید
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # مقداردهی اولیه مسیر تصویر (توجه کنید که نوع فایل باید png باشد)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # مدل‌های gpt-image تصویر را به صورت base64 (b64_json) برمی‌گردانند، نه URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # نمایش تصویر در نمایشگر تصویر پیش‌فرض
        image = Image.open(image_path)
        image.show()

    # گرفتن استثناها
    except BadRequestError as err:
        print(err)

    ```

بیایید این کد را توضیح دهیم:

- ابتدا کتابخانه‌های مورد نیاز از جمله کتابخانه OpenAI، کتابخانه dotenv، ماژول base64 و کتابخانه Pillow را وارد می‌کنیم.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- سپس، متغیرهای محیطی را از فایل *.env* بارگذاری می‌کنیم.

    ```python
    # وارد کردن dotenv
    dotenv.load_dotenv()
    ```

- بعد از آن، کلاینت Azure OpenAI (Microsoft Foundry) را پیکربندی می‌کنیم.

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- در مرحله بعد، تصویر را تولید می‌کنیم:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # متن درخواست خود را اینجا وارد کنید
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    مدل‌های `gpt-image` تصویر را به صورت رشته **base64** در `data[0].b64_json` برمی‌گردانند. ما آن را به بایت تبدیل کرده و در یک فایل می‌نویسیم — لینک دانلود وجود ندارد.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- در نهایت، تصویر را باز کرده و با استفاده از نمایشگر تصویر استاندارد آن را نمایش می‌دهیم:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### جزئیات بیشتر درباره تولید تصویر

بیایید پارامترهای `images.generate` را بررسی کنیم:

- **prompt**، متن توضیحی است که برای تولید تصویر استفاده می‌شود. در اینجا متن "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils" است.
- **size**، اندازه تصویر تولید شده است (مثلاً `1024x1024`، `1536x1024`، `1024x1536` یا `"auto"`).
- **n**، تعداد تصاویر تولید شده است. در اینجا یک تصویر تولید می‌کنیم.
- **model**، نام استقرار مدل تصویر شما است (مثلاً `gpt-image-2`).

> مدل‌های تصویر پارامتر `temperature` را نمی‌گیرند — این پارامتر برای کنترل تولید متن است. برای ایجاد تنوع، دوباره API را فراخوانی کنید؛ برای کاهش تنوع، توضیح خود را دقیق‌تر کنید.

## توانایی‌های اضافی در تولید تصویر

شما دیدید که چگونه می‌توان با چند خط کد پایتون یک تصویر تولید کرد. مدل‌های `gpt-image` همچنین می‌توانند یک تصویر موجود را **ویرایش** کنند. با ارائه یک تصویر، یک **ماسک** اختیاری (که بخش مورد تغییر را مشخص می‌کند) و یک توضیح، می‌توانید قسمتی از تصویر را تغییر دهید — مثلاً به خرگوش خود کلاهی اضافه کنید.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# ویرایش‌ها نیز به صورت base64 بازگردانده می‌شوند
edited_image = base64.b64decode(response.data[0].b64_json)
```

تصویر پایه فقط شامل خرگوش است؛ تصویر نهایی کلاه را روی خرگوش دارد.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
